# Top-5 Leagues Player Stats — ETL & Star Schema Build

**Pipeline:** Extract (read raw CSVs) → Transform (clean, standardize, engineer features, build a star schema) → ready to Load into Azure.



In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


## 1. Load raw data

Use a **relative path** instead of a hardcoded local path, so the notebook runs on any machine
(including when you move it to a repo, a container, or later an Azure compute instance).
Put both CSVs in a `data/` folder next to this notebook, or change `RAW_DIR` below.


In [5]:
RAW_DIR = Path("raw data")  # <-- adjust this if your CSVs live elsewhere

df1 = pd.read_csv(RAW_DIR / "Top5_League_Players_2017to2024_dataset.csv", sep=";", decimal=",")
df2 = pd.read_csv(RAW_DIR / "players_data-2025_2026.csv", decimal=",")

print("df1 (2017-2024):", df1.shape)
print("df2 (2025-2026):", df2.shape)


df1 (2017-2024): (22929, 178)
df2 (2025-2026): (2839, 102)


In [6]:
df1.head(5)

,league,season,team,player,nation_,pos_,age_,born_,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Expected_xG,Expected_npxG,Expected_xAG,Expected_npxG+xAG,Progression_PrgC,Progression_PrgP,Progression_PrgR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,Per 90 Minutes_xG,Per 90 Minutes_xAG,Per 90 Minutes_xG+xAG,Per 90 Minutes_npxG,Per 90 Minutes_npxG+xAG,90s_,Expected_G-xG,Expected_np:G-xG,Expected_npxG/Sh,Standard_Dist,Standard_FK,Standard_G/Sh,Standard_G/SoT,Standard_Gls,Standard_PK,Standard_PKatt,Standard_Sh,Standard_Sh/90,Standard_SoT,Standard_SoT%,Standard_SoT/90,1/3_,Ast_,CrsPA_,Expected_A-xAG,Expected_xA,KP_,Long_Att,Long_Cmp,Long_Cmp%,Medium_Att,Medium_Cmp,Medium_Cmp%,PPA_,PrgP_,Short_Att,Short_Cmp,Short_Cmp%,Total_Att,Total_Cmp,Total_Cmp%,Total_PrgDist,Total_TotDist,xAG_,Att_,Corner Kicks_In,Corner Kicks_Out,Corner Kicks_Str,Outcomes_Blocks,Outcomes_Cmp,Outcomes_Off,Pass Types_CK,Pass Types_Crs,Pass Types_Dead,Pass Types_FK,Pass Types_Live,Pass Types_Sw,Pass Types_TB,Pass Types_TI,GCA Types_Def,GCA Types_Fld,GCA Types_PassDead,GCA Types_PassLive,GCA Types_Sh,GCA Types_TO,GCA_GCA,GCA_GCA90,SCA Types_Def,SCA Types_Fld,SCA Types_PassDead,SCA Types_PassLive,SCA Types_Sh,SCA Types_TO,SCA_SCA,SCA_SCA90,Blocks_Blocks,Blocks_Pass,Blocks_Sh,Challenges_Att,Challenges_Lost,Challenges_Tkl,Challenges_Tkl%,Clr_,Err_,Int_,Tackles_Att 3rd,Tackles_Def 3rd,Tackles_Mid 3rd,Tackles_Tkl,Tackles_TklW,Tkl+Int_,Carries_1/3,Carries_CPA,Carries_Carries,Carries_Dis,Carries_Mis,Carries_PrgC,Carries_PrgDist,Carries_TotDist,Receiving_PrgR,Receiving_Rec,Take-Ons_Att,Take-Ons_Succ,Take-Ons_Succ%,Take-Ons_Tkld,Take-Ons_Tkld%,Touches_Att 3rd,Touches_Att Pen,Touches_Def 3rd,Touches_Def Pen,Touches_Live,Touches_Mid 3rd,Touches_Touches,Playing Time_Min%,Playing Time_Mn/MP,Starts_Compl,Starts_Mn/Start,Starts_Starts,Subs_Mn/Sub,Subs_Subs,Subs_unSub,Team Success (xG)_On-Off,Team Success (xG)_onxG,Team Success (xG)_onxGA,Team Success (xG)_xG+/-,Team Success (xG)_xG+/-90,Team Success_+/-,Team Success_+/-90,Team Success_On-Off,Team Success_PPM,Team Success_onG,Team Success_onGA,Aerial Duels_Lost,Aerial Duels_Won,Aerial Duels_Won%,Performance_2CrdY,Performance_Crs,Performance_Fld,Performance_Fls,Performance_Int,Performance_OG,Performance_Off,Performance_PKcon,Performance_PKwon,Performance_Recov,Performance_TklW
0,ENG-Premier League,1718,Arsenal,Aaron Ramsey,WAL,MF,26.0,1990.0,24,21,1846,20.5,7,8,15,7,0,0,0,0,6.1,6.1,5.4,11.5,61.0,134.0,161.0,0.34,0.39,0.73,0.34,0.73,0.30,0.26,0.56,0.30,0.56,20.5,0.9,0.9,0.11,16.6,0.0,0.13,0.28,7,0,0,56.0,2.73,25,44.6,1.22,102.0,8,3.0,2.6,5.3,28.0,97.0,69.0,71.1,457.0,390.0,85.3,36.0,134.0,627.0,556.0,88.7,1277.0,1065.0,83.4,4329.0,16341.0,5.4,1277.0,0.0,0.0,0.0,24.0,1065.0,9.0,0.0,17.0,32.0,20.0,1236.0,12.0,10.0,12.0,0.0,2.0,0.0,10.0,0.0,1.0,13.0,0.63,0.0,3.0,0.0,53.0,3.0,4.0,63.0,3.07,14.0,11.0,3.0,44.0,36.0,8.0,18.2,6.0,1.0,24.0,0.0,18.0,17.0,35.0,27.0,59.0,54.0,18.0,1082.0,37.0,49.0,61.0,2709.0,5745.0,161.0,1214.0,36.0,29.0,80.6,7.0,19.4,552.0,91.0,174.0,16.0,1480.0,770.0,1480.0,54.0,77,16.0,NaN,21,NaN,3,0,0.93,42.8,23.0,19.8,0.97,17.0,0.83,0.49,1.83,42.0,25.0,15.0,10.0,40.0,0.0,17.0,33.0,28,24.0,0.0,0.0,0.0,2.0,128.0,27.0
1,ENG-Premier League,1718,Arsenal,Ainsley Maitland-Niles,ENG,"DF,MF",19.0,1997.0,15,8,914,10.2,0,0,0,0,0,0,1,0,0.2,0.2,0.9,1.1,21.0,35.0,58.0,0.00,0.00,0.00,0.00,0.00,0.02,0.08,0.11,0.02,0.11,10.2,-0.2,-0.2,0.05,20.9,0.0,0.00,NaN,0,0,0,5.0,0.49,0,0.0,0.00,37.0,0,4.0,-0.9,1.4,8.0,59.0,42.0,71.2,215.0,178.0,82.8,14.0,35.0,321.0,290.0,90.3,640.0,530.0,82.8,2475.0,8182.0,0.9,640.0,0.0,0.0,0.0,18.0,530.0,3.0,0.0,20.0,87.0,6.0,550.0,3.0,0.0,81.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,14.0,0.0,4.0,18.0,1.77,15.0,14.0,1.0,31.0,17.0,14.0,45.2,35.0,2.0,12.0,6.0,12.0,7.0,25.0,17.0,37.0,18.0,5.0,425.0,14.0,18.

In [7]:
df2.head(5)

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,G+A-PK,Rk_stats_keeper,Nation_stats_keeper,Pos_stats_keeper,Comp_stats_keeper,Age_stats_keeper,Born_stats_keeper,MP_stats_keeper,Starts_stats_keeper,Min_stats_keeper,90s_stats_keeper,GA,GA90,SoTA,Saves,Save%,W,D,L,CS,CS%,PKatt_stats_keeper,PKA,PKsv,PKm,Rk_stats_shooting,Nation_stats_shooting,Pos_stats_shooting,Comp_stats_shooting,Age_stats_shooting,Born_stats_shooting,90s_stats_shooting,Gls_stats_shooting,Sh,SoT,SoT%,Sh/90,SoT/90,G/Sh,G/SoT,PK_stats_shooting,PKatt_stats_shooting,Rk_stats_playing_time,Nation_stats_playing_time,Pos_stats_playing_time,Comp_stats_playing_time,Age_stats_playing_time,Born_stats_playing_time,MP_stats_playing_time,Min_stats_playing_time,Mn/MP,Min%,90s_stats_playing_time,Starts_stats_playing_time,Mn/Start,Compl,Subs,Mn/Sub,unSub,PPM,onG,onGA,+/-,+/-90,On-Off,Rk_stats_misc,Nation_stats_misc,Pos_stats_misc,Comp_stats_misc,Age_stats_misc,Born_stats_misc,90s_stats_misc,CrdY_stats_misc,CrdR_stats_misc,2CrdY,Fls,Fld,Off,Crs,Int,TklW,OG
0,1,Brenden Aaronson,us USA,"MF,FW",Leeds United,eng Premier League,24.0,2000.0,37,30,2449,27.2,4,5,9,4,0,0,3,0,0.33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,us USA,"MF,FW",eng Premier League,24.0,2000.0,27.2,4,47,17,36.2,1.73,0.62,0.09,0.24,0,0,1,us USA,"MF,FW",eng Premier League,24.0,2000.0,37,2449,66,71.6,27.2,30,77.0,2,7,21.0,1,1.19,35,31,4,+0.15,+1.17,1,us USA,"MF,FW",eng Premier League,24.0,2000.0,27.2,3,0,0,20,51,5,37,17,27,0
1,2,Jerome Abbey,eng ENG,MF,Wolves,eng Premier League,15.0,2009.0,1,0,17,0.2,0,0,0,0,0,0,0,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,eng ENG,MF,eng Premier League,15.0,2009.0,0.2,0,0,0,NaN,0.00,0.00,NaN,NaN,0,0,2,eng ENG,MF,eng Premier League,15.0,2009.0,1,17,17,0.5,0.2,0,NaN,0,1,17.0,1,1.00,0,0,0,0.00,+1.08,2,eng ENG,MF,eng Premier League,15.0,2009.0,0.2,0,0,0,0,0,1,0,1,0,0
2,3,Zach Abbott,eng ENG,DF,Nottingham Forest,eng Premier League,19.0,2006.0,3,2,164,1.8,0,0,0,0,0,0,0,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,eng ENG,DF,eng Premier League,19.0,2006.0,1.8,0,0,0,NaN,0.00,0.00,NaN,NaN,0,0,3,eng ENG,DF,eng Premier League,19.0,2006.0,3,164,55,4.8,1.8,2,60.0,0,1,45.0,13,1.00,2,4,-2,-1.10,-1.07,3,eng ENG,DF,eng Premier League,19.0,2006.0,1.8,0,0,0,3,1,0,3,2,4,0
3,4,Jones El-Abdellaoui,ma MAR,"MF,FW",Celta Vigo,es La Liga,19.0,2006.0,22,4,681,7.6,2,0,2,2,0,0,0,0,0.26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,ma MAR,"MF,FW",es La Liga,19.0,2006.0,7.6,2,17,4,23.5,2.25,0.53,0.12,0.50,0,0,5,ma MAR,"MF,FW",es La Liga,19.0,2006.0,22,681,31,19.9,7.6,4,75.0,1,18,21.0,16,1.55,17,12,5,+0.66,+0.66,4,ma MAR,"MF,FW",es La Liga,19.0,2006.0,7.6,0,0,0,7,3,2,30,3,3,0
4,5,Himad Abdelli,dz ALG,MF,Marseille,fr Ligue 1,25.0,1999.0,8,1,138,1.5,0,0,0,0,0,0,1,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,dz ALG,MF,fr Ligue 1,25.0,1999.0,1.5,0,4,1,25.0,2.61,0.65,0.00,0.00,0,0,6,dz ALG,MF,fr Ligue 1,25.0,1999.0,8,138,17,4.5,1.5,1,58.0,0,7,11.0,2,1.00,2,8,-6,-3.91,-4.65,5,dz ALG,MF,fr Ligue 1,25.0,1999.0,1.5,1,0,0,3,0,0,1,2,1,0


## 2. Standardize `season` and `league` labels




In [8]:
def expand_season_code(code_value):
    """Convert a compact season code like 1718 into '2017-2018'."""
    s = str(int(code_value))
    year_start = int(s[:2])
    year_end = int(s[2:])
    return f"20{year_start:02d}-20{year_end:02d}"

df1["season"] = df1["season"].apply(expand_season_code)
df2["season"] = "2025-2026"

LEAGUE_NAME_MAP = {
    "ENG-Premier League": "Premier League",
    "ESP-La Liga": "La Liga",
    "FRA-Ligue 1": "Ligue 1",
    "GER-Bundesliga": "Bundesliga",
    "ITA-Serie A": "Serie A",
    "eng Premier League": "Premier League",
    "es La Liga": "La Liga",
    "fr Ligue 1": "Ligue 1",
    "it Serie A": "Serie A",
    "de Bundesliga": "Bundesliga",
}

df1["league"] = df1["league"].map(LEAGUE_NAME_MAP).fillna(df1["league"])
df2["Comp"] = df2["Comp"].map(LEAGUE_NAME_MAP).fillna(df2["Comp"])

print(sorted(df1["season"].unique()))
print(sorted(df1["league"].unique()))
print(sorted(df2["Comp"].unique()))


['2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025']
['Bundesliga', 'La Liga', 'Ligue 1', 'Premier League', 'Serie A']
['Bundesliga', 'La Liga', 'Ligue 1', 'Premier League', 'Serie A']


## 3. Rename columns to a common schema and merge

Same idea as before, with one addition: `"Playing Time_90s"` in df1 is now mapped to
`Full_Match_Equivalents`, which was missing in the previous version (so old seasons had
`NaN` there after the merge).


In [9]:
df1 = df1.rename(columns={
    "player": "player",
    "team": "team",
    "league": "league",
    "season": "season",
    "pos_": "Pos",
    "age_": "Age",
    "nation_": "Nation",
    "born_": "Born",

    "Playing Time_MP": "Matches_Played",
    "Playing Time_Starts": "Matches_Started",
    "Playing Time_Min": "Minutes_Played",
    "Playing Time_90s": "Full_Match_Equivalents",

    "Performance_Gls": "Goals",
    "Performance_Ast": "Assists",
    "Performance_G+A": "Goal_Contributions",

    "Expected_xG": "Expected_Goals",
    "Expected_xAG": "Expected_Assists",
    "Expected_npxG": "NonPenalty_Expected_Goals",

    "Standard_Sh": "Shots",
    "Standard_SoT": "Shots_On_Target",

    "Progression_PrgC": "Progressive_Carries",
    "Progression_PrgP": "Progressive_Passes",

    "KP_": "Key_Passes",
    "Total_Att": "Pass_Attempts",
    "Total_Cmp": "Pass_Completed",

    "SCA_SCA": "Shot_Creating_Actions",
    "GCA_GCA": "Goal_Creating_Actions",

    "Tackles_Tkl": "Tackles",
    "Int_": "Interceptions",
    "Blocks_Blocks": "Blocks",

    "Carries_Carries": "Carries",
    "Touches_Touches": "Touches",
})

df2 = df2.rename(columns={
    "Player": "player",
    "Squad": "team",
    "Comp": "league",
    "Pos": "Pos",
    "Age": "Age",
    "Nation": "Nation",
    "Born": "Born",
    "MP": "Matches_Played",
    "Starts": "Matches_Started",
    "Min": "Minutes_Played",
    "90s": "Full_Match_Equivalents",
    "Gls": "Goals",
    "Ast": "Assists",
    "G+A": "Goal_Contributions",
    # NOTE: df2 (2025-2026) does NOT contain xG, Sh, SoT, PrgC, PrgP, KP,
    # SCA, GCA, Tkl, Blocks, Carries, or Touches at all. Renaming them here
    # would do nothing (pandas silently ignores renames for columns that
    # don't exist) -- see the data-completeness check in the next section.
})

df = pd.concat([df1, df2], ignore_index=True, sort=False)
print(df.shape)


(25768, 266)


## 4. Select analysis columns (comma bug fixed) + check data completeness

The original list was missing commas after `"season"`, `"Touches"`, and `"Carries"`.
In Python, adjacent string literals with no comma between them get silently concatenated
into one string (e.g. `"season" "Matches_Played"` becomes `"seasonMatches_Played"`), so those
items quietly stopped being valid column names and vanished from the selection. That's what
caused the `KeyError: 'season'` later in the notebook.


In [10]:
key_cols = [
    "player", "team", "league", "season", "Pos", "Age", "Nation", "Born",

    "Matches_Played", "Matches_Started", "Minutes_Played", "Full_Match_Equivalents",

    "Goals", "Assists", "Goal_Contributions",

    "Expected_Goals", "Expected_Assists", "NonPenalty_Expected_Goals",

    "Shots", "Shots_On_Target", "Progressive_Carries", "Progressive_Passes", "Touches",

    "Key_Passes", "Pass_Attempts", "Pass_Completed", "Shot_Creating_Actions", "Carries",

    "Goal_Creating_Actions", "Tackles", "Interceptions", "Blocks",
]

available_key_cols = [c for c in key_cols if c in df.columns]
missing_cols = [c for c in key_cols if c not in df.columns]
print("Missing from merged df:", missing_cols)

df_analysis = df[available_key_cols].copy()
df_analysis.info()


Missing from merged df: []
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25768 entries, 0 to 25767
Data columns (total 32 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   player                     25768 non-null  object 
 1   team                       25768 non-null  object 
 2   league                     25768 non-null  object 
 3   season                     25768 non-null  object 
 4   Pos                        25767 non-null  object 
 5   Age                        25763 non-null  object 
 6   Nation                     25755 non-null  object 
 7   Born                       25763 non-null  float64
 8   Matches_Played             25768 non-null  int64  
 9   Matches_Started            25768 non-null  int64  
 10  Minutes_Played             25768 non-null  int64  
 11  Full_Match_Equivalents     25768 non-null  object 
 12  Goals                      25768 non-null  int64  
 13  Assists            

In [11]:
# Data-completeness check by season: shows which columns are structurally
# missing for the 2025-2026 season because the source file doesn't have them.
completeness = df_analysis.groupby("season").apply(lambda g: g.isna().mean())
completeness.T.style.background_gradient(cmap="Reds", axis=1)


season,2017-2018,2018-2019,2019-2020,2020-2021,2021-2022,2022-2023,2023-2024,2024-2025,2025-2026
player,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
team,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
league,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
season,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Pos,0.000312,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Age,0.000312,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000701,0.000704
Nation,0.000625,0.000753,0.000366,0.000354,0.000342,0.000000,0.000351,0.000701,0.001057
Born,0.000312,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000701,0.000704
Matches_Played,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Matches_Started,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## 5. Clean and type-cast numeric columns

**Important fix:** the old code ran a digits-only regex over *every* text column, which also hit
`player`, `team`, `league`, `Nation` and stripped the letters out of them (turning names into
empty strings), and was almost certainly what caused the `MemoryError` too. The regex should only
touch columns that are numbers stored as text (like `Age` in a `"25-060"` style format), not
identity/text columns.


In [12]:
TEXT_ID_COLS = ["player", "team", "league", "season", "Pos", "Nation"]

NUMERIC_TEXT_COLS = [
    c for c in df_analysis.columns
    if c not in TEXT_ID_COLS and df_analysis[c].dtype == "object"
]
print("Columns being numeric-cleaned:", NUMERIC_TEXT_COLS)

for col in NUMERIC_TEXT_COLS:
    df_analysis[col] = (
        df_analysis[col]
        .astype(str)
        .str.replace(r'[^\d.-]', '', regex=True)
        .replace('', np.nan)
    )
    df_analysis[col] = pd.to_numeric(df_analysis[col], errors="coerce")

df_analysis["Minutes_Played"] = pd.to_numeric(df_analysis["Minutes_Played"], errors="coerce").fillna(0)

int_cols = [
    "Shots", "Shots_On_Target", "Progressive_Carries", "Progressive_Passes",
    "Pass_Attempts", "Pass_Completed", "Shot_Creating_Actions",
    "Tackles", "Interceptions", "Blocks",
]
int_cols = [c for c in int_cols if c in df_analysis.columns]
df_analysis[int_cols] = df_analysis[int_cols].round().astype("Int64")

df_analysis.dtypes


Columns being numeric-cleaned: ['Age', 'Full_Match_Equivalents']


player                        object
team                          object
league                        object
season                        object
Pos                           object
Age                          float64
Nation                        object
Born                         float64
Matches_Played                 int64
Matches_Started                int64
Minutes_Played                 int64
Full_Match_Equivalents       float64
Goals                          int64
Assists                        int64
Goal_Contributions             int64
Expected_Goals               float64
Expected_Assists             float64
NonPenalty_Expected_Goals    float64
Shots                          Int64
Shots_On_Target                Int64
Progressive_Carries            Int64
Progressive_Passes             Int64
Touches                      float64
Key_Passes                   float64
Pass_Attempts                  Int64
Pass_Completed                 Int64
Shot_Creating_Actions          Int64
C

## 6. Feature engineering

Now that everything numeric is actually numeric, the derived features are computed safely.


In [13]:
df_analysis["xg_diff"] = df_analysis["Goals"] - df_analysis["Expected_Goals"]
df_analysis["xa_diff"] = df_analysis["Assists"] - df_analysis["Expected_Assists"]
df_analysis["progressive_actions"] = df_analysis["Progressive_Carries"] + df_analysis["Progressive_Passes"]
df_analysis["defensive_index"] = df_analysis["Tackles"] + df_analysis["Interceptions"] + df_analysis["Blocks"]

df_analysis.head()


,player,team,league,season,Pos,Age,Nation,Born,Matches_Played,Matches_Started,Minutes_Played,Full_Match_Equivalents,Goals,Assists,Goal_Contributions,Expected_Goals,Expected_Assists,NonPenalty_Expected_Goals,Shots,Shots_On_Target,Progressive_Carries,Progressive_Passes,Touches,Key_Passes,Pass_Attempts,Pass_Completed,Shot_Creating_Actions,Carries,Goal_Creating_Actions,Tackles,Interceptions,Blocks,xg_diff,xa_diff,progressive_actions,defensive_index
0,Aaron Ramsey,Arsenal,Premier League,2017-2018,MF,26.0,WAL,1990.0,24,21,1846,20.5,7,8,15,6.1,5.4,6.1,56,25,61,134,1480.0,28.0,1277,1065,63,1082.0,13.0,35,24,14,0.9,2.6,195,73
1,Ainsley Maitland-Niles,Arsenal,Premier League,2017-2018,"DF,MF",19.0,ENG,1997.0,15,8,914,10.2,0,0,0,0.2,0.9,0.2,5,0,21,35,762.0,8.0,640,530,18,425.0,0.0,25,12,15,-0.2,-0.9,56,52
2,Alex Iwobi,Arsenal,Premier League,2017-2018,"MF,FW",21.0,NGA,1996.0,26,22,1830,20.3,3,5,8,3.9,3.5,3.9,45,22,93,138,1358.0,36.0,1171,997,82,1058.0,12.0,19,7,19,-0.9,1.5,231,45
3,Alex Oxlade-Chamberlain,Arsenal,Premier League,2017-2018,DF,23.0,ENG,1993.0,3,3,241,2.7,0,0,0,0.3,0.5,0.3,9,2,19,16,203.0,5.0,158,124,13,125.0,0.0,5,6,5,-0.3,-0.5,35,16
4,Alexandre Lacazette,Arsenal,Premier League,2017-2018,FW,26.0,FRA,1991.0,32,26,2202,24.5,14,4,18,13.7,4.4,12.1,66,32,39,59,980.0,36.0,744,558,95,621.0,13.0,24,7,19,0.3,-0.4,98,50


## 7. Build a star schema (dimension + fact tables)


In [14]:
# --- Dimension: Player ---
dim_player = (
    df_analysis[["player", "Nation", "Born"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_player.insert(0, "player_id", dim_player.index + 1)

# --- Dimension: Team ---
dim_team = pd.DataFrame({"team": df_analysis["team"].dropna().unique()})
dim_team.insert(0, "team_id", dim_team.index + 1)

# --- Dimension: League ---
dim_league = pd.DataFrame({"league": df_analysis["league"].dropna().unique()})
dim_league.insert(0, "league_id", dim_league.index + 1)

# --- Dimension: Season ---
dim_season = pd.DataFrame({"season": df_analysis["season"].dropna().unique()})
dim_season.insert(0, "season_id", dim_season.index + 1)

for name, dim in [("dim_player", dim_player), ("dim_team", dim_team),
                   ("dim_league", dim_league), ("dim_season", dim_season)]:
    print(name, dim.shape)


dim_player (9876, 4)
dim_team (161, 2)
dim_league (5, 2)
dim_season (9, 2)


In [15]:
# --- Fact table: join surrogate keys back onto the analysis table ---
fact_player_stats = df_analysis.merge(
        dim_player, on=["player", "Nation", "Born"], how="left"
    ).merge(
        dim_team, on="team", how="left"
    ).merge(
        dim_league, on="league", how="left"
    ).merge(
        dim_season, on="season", how="left"
    )

# Drop the natural-key text columns now that we have surrogate keys
# (they still live in the dimension tables)
fact_player_stats = fact_player_stats.drop(columns=["player", "team", "league", "season", "Nation", "Born"])

# Reorder so the keys come first
key_id_cols = ["player_id", "team_id", "league_id", "season_id"]
fact_player_stats = fact_player_stats[key_id_cols + [c for c in fact_player_stats.columns if c not in key_id_cols]]

print(fact_player_stats.shape)
fact_player_stats.head()


(25768, 34)


,player_id,team_id,league_id,season_id,Pos,Age,Matches_Played,Matches_Started,Minutes_Played,Full_Match_Equivalents,Goals,Assists,Goal_Contributions,Expected_Goals,Expected_Assists,NonPenalty_Expected_Goals,Shots,Shots_On_Target,Progressive_Carries,Progressive_Passes,Touches,Key_Passes,Pass_Attempts,Pass_Completed,Shot_Creating_Actions,Carries,Goal_Creating_Actions,Tackles,Interceptions,Blocks,xg_diff,xa_diff,progressive_actions,defensive_index
0,1,1,1,1,MF,26.0,24,21,1846,20.5,7,8,15,6.1,5.4,6.1,56,25,61,134,1480.0,28.0,1277,1065,63,1082.0,13.0,35,24,14,0.9,2.6,195,73
1,2,1,1,1,"DF,MF",19.0,15,8,914,10.2,0,0,0,0.2,0.9,0.2,5,0,21,35,762.0,8.0,640,530,18,425.0,0.0,25,12,15,-0.2,-0.9,56,52
2,3,1,1,1,"MF,FW",21.0,26,22,1830,20.3,3,5,8,3.9,3.5,3.9,45,22,93,138,1358.0,36.0,1171,997,82,1058.0,12.0,19,7,19,-0.9,1.5,231,45
3,4,1,1,1,DF,23.0,3,3,241,2.7,0,0,0,0.3,0.5,0.3,9,2,19,16,203.0,5.0,158,124,13,125.0,0.0,5,6,5,-0.3,-0.5,35,16
4,5,1,1,1,FW,26.0,32,26,2202,24.5,14,4,18,13.7,4.4,12.1,66,32,39,59,980.0,36.0,744,558,95,621.0,13.0,24,7,19,0.3,-0.4,98,50


In [16]:
# Sanity check: every fact row should have matched to a valid dimension row (no NaN keys)
print(fact_player_stats[key_id_cols].isna().sum())


player_id    0
team_id      0
league_id    0
season_id    0
dtype: int64


## 8. Export each table to its own file




In [17]:
OUT_DIR = Path("processed")
OUT_DIR.mkdir(exist_ok=True)

dim_player.to_csv(OUT_DIR / "dim_player.csv", index=False)
dim_team.to_csv(OUT_DIR / "dim_team.csv", index=False)
dim_league.to_csv(OUT_DIR / "dim_league.csv", index=False)
dim_season.to_csv(OUT_DIR / "dim_season.csv", index=False)
fact_player_stats.to_csv(OUT_DIR / "fact_player_stats.csv", index=False)

print("Exported:", [f.name for f in OUT_DIR.glob("*.csv")])


Exported: ['dim_league.csv', 'dim_player.csv', 'dim_season.csv', 'dim_team.csv', 'fact_player_stats.csv']
